In [1]:

!pip install fpdf nltk -q

You should consider upgrading via the '/Users/binitapandey/phishing-detection-ml-system/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:

import pandas as pd
import numpy as np
import re
import nltk
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
import os
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("All libraries loaded!")

All libraries loaded!


In [3]:

DATASET_PATH = '/content/drive/MyDrive/8th sem major project/sms_scam_detection_dataset_merged_with_lang1.csv'

df = pd.read_csv(DATASET_PATH)
print(f" Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/8th sem major project/sms_scam_detection_dataset_merged_with_lang1.csv'

In [ ]:

def clean_text(text):
    """
    Clean SMS text - ONLY clean the message, DO NOT replace URL, Email, Phone
    Keep all original content, just make it readable
    """
    if pd.isna(text):
        return ""

    text = str(text)

    abbreviations = {
        'u': 'you', 'r': 'are', 'ur': 'your', 'urs': 'yours',
        'd': 'the', 'y': 'why', 'b': 'be', 'c': 'see', 'k': 'okay',
        'n': 'and', 'w': 'with', 'w/o': 'without', 'w/': 'with',
        '4': 'for', '2': 'to', '2day': 'today', '2moro': 'tomorrow',
        '2nite': 'tonight', 'b4': 'before', 'pls': 'please',
        'plz': 'please', 'thx': 'thanks', 'ty': 'thank you',
        'lol': 'laugh out loud', 'brb': 'be right back',
        'btw': 'by the way', 'idk': 'i do not know',
        'imo': 'in my opinion', 'irl': 'in real life',
        'jk': 'just kidding', 'lmk': 'let me know',
        'np': 'no problem', 'omg': 'oh my god',
        'gtg': 'got to go', 'tbh': 'to be honest',
        'dm': 'direct message', 'pm': 'private message',
        'asap': 'as soon as possible', 'afk': 'away from keyboard',
        'bday': 'birthday', 'cya': 'see you', 'fyi': 'for your information',
        'hmu': 'hit me up', 'ily': 'i love you',
        'lmao': 'laughing my ass off', 'rofl': 'rolling on floor laughing',
        'smh': 'shaking my head', 'ttyl': 'talk to you later',
        'wyd': 'what are you doing', 'wtf': 'what the fuck',
        'yk': 'you know', 'yw': 'you\'re welcome',
        'omw': 'on my way', 'brb': 'be right back',
        'gtg': 'got to go', 'cya': 'see you',
        'da': 'the', 'ya': 'you', 'yall': 'you all',
        'l8r': 'later', 'gr8': 'great', 'g2g': 'got to go',
        'bbl': 'be back later', 'bbs': 'be back soon',
        'msg': 'message', 'txt': 'text', 'fav': 'favorite',
        'sec': 'second', 'mins': 'minutes', 'hrs': 'hours',
        'ppl': 'people', 'bc': 'because', 'cuz': 'because',
        'tbh': 'to be honest', 'idc': 'i do not care',
        'ily': 'i love you', 'ilu': 'i love you',
    }

    for short, full in sorted(abbreviations.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(r'\b' + re.escape(short) + r'\b', full, text, flags=re.IGNORECASE)

    text = re.sub(r'\s+([!?.])', r'\1', text)
    text = re.sub(r'([!?.])\s+', r'\1 ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'([!?.])\1+', r'\1', text)

    text = re.sub(r'\.{2,}', '.', text)

    text = text.lower()

    return text

print("🔄 Cleaning messages...")
df['cleaned_message'] = df['text'].apply(clean_text)

df['label'] = df['label'].str.lower()

df_clean = df[df['cleaned_message'].str.len() > 2].copy()
df_clean = df_clean.dropna(subset=['cleaned_message'])
df_clean = df_clean.drop_duplicates(subset=['cleaned_message'])

print(f" Cleaning complete!")
print(f"Original: {len(df)} | Cleaned: {len(df_clean)}")
print(f"Removed: {len(df) - len(df_clean)} rows")

print("\n📊 Sample of cleaned data:")
display(df_clean[['label', 'cleaned_message']].head(10))

print("\n" + "="*60)
print("🔍 CLEANING EXAMPLES (URL, Email, Phone KEPT AS IS):")
print("="*60)

test_texts = [
    "Call me on 9854260260",
    "Visit my website http://example.com",
    "Email me at test@gmail.com",
    "My number is +91-9876543210",
    "Check www.google.com for more info",
    "Contact 98765 43210 or 98765-43210",
    "PLZ call me 2day at 1234567890",
    "Visit https://www.example.com/page for details"
]

for test in test_texts:
    cleaned = clean_text(test)
    print(f"Original: {test}")
    print(f"Cleaned:  {cleaned}")
    print("-"*50)

🔄 Cleaning messages...
✅ Cleaning complete!
Original: 139495 | Cleaned: 9841
Removed: 129654 rows

📊 Sample of cleaned data:


,label,cleaned_message
0,ham,your opinion about me? 1. over to. jada 3. kus...
1,ham,what's up? do you want me to come online? if y...
2,ham,so you workin overtime nigpun?
3,ham,"also sir, i sent you an email about how to log..."
4,spam,please stay at home. to encourage the notion o...
5,spam,bankofamerica alert 137943. please follow http...
6,ham,sorry dude. dont know how i forgot. even after...
7,ham,i don't quite know what to do. i still can't g...
8,ham,ok lor. anyway i thk we cant get tickets now c...
9,ham,wat are you doing now?



🔍 CLEANING EXAMPLES (URL, Email, Phone KEPT AS IS):
Original: Call me on 9854260260
Cleaned:  call me on 9854260260
--------------------------------------------------
Original: Visit my website http://example.com
Cleaned:  visit my website http://example.com
--------------------------------------------------
Original: Email me at test@gmail.com
Cleaned:  email me at test@gmail.com
--------------------------------------------------
Original: My number is +91-9876543210
Cleaned:  my number is +91-9876543210
--------------------------------------------------
Original: Check www.google.com for more info
Cleaned:  check www.google.com for more info
--------------------------------------------------
Original: Contact 98765 43210 or 98765-43210
Cleaned:  contact 98765 43210 or 98765-43210
--------------------------------------------------
Original: PLZ call me 2day at 1234567890
Cleaned:  please call me today at 1234567890
--------------------------------------------------
Original: Visit ht

In [ ]:

def clean_text_for_pdf(text):
    """Clean text for PDF - remove problematic Unicode characters"""
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace('\ufffd', '')  
    text = text.replace('\u2018', "'") 
    text = text.replace('\u2019', "'")  
    text = text.replace('\u201c', '"')  
    text = text.replace('\u201d', '"')  
    text = text.replace('\u2013', '-')  
    text = text.replace('\u2014', '-')  
    text = text.replace('\u2026', '...') 
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text

def create_pdf_report(df_clean, original_df, filename='cleaning_report.pdf'):
    """
    Generate a comprehensive PDF report with all cleaning details
    """
    print("📄 Generating PDF report...")

    total_original = len(original_df)
    total_cleaned = len(df_clean)
    removed = total_original - total_cleaned
    removed_percent = (removed / total_original) * 100 if total_original > 0 else 0

    spam_count = len(df_clean[df_clean['label'] == 'spam'])
    ham_count = len(df_clean[df_clean['label'] == 'ham'])
    spam_percent = (spam_count / total_cleaned) * 100 if total_cleaned > 0 else 0
    ham_percent = (ham_count / total_cleaned) * 100 if total_cleaned > 0 else 0

    avg_words = df_clean['cleaned_message'].str.split().str.len().mean()
    avg_chars = df_clean['cleaned_message'].str.len().mean()
    max_words = df_clean['cleaned_message'].str.split().str.len().max()
    min_words = df_clean['cleaned_message'].str.split().str.len().min()

    has_url = df_clean['cleaned_message'].str.contains(r'https?://|www\.', case=False, na=False).sum()
    has_email = df_clean['cleaned_message'].str.contains(r'@', na=False).sum()
    has_phone = df_clean['cleaned_message'].str.contains(r'\d{3}[-.]?\d{3}[-.]?\d{4}', na=False).sum()

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Arial', 'B', 24)
    pdf.cell(0, 20, 'SMS Spam Detection', ln=True, align='C')
    pdf.set_font('Arial', 'B', 18)
    pdf.cell(0, 15, 'Data Cleaning Report', ln=True, align='C')
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 10, f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True, align='C')
    pdf.ln(15)

    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '1. Dataset Overview', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"Total Original Messages: {total_original:,}", ln=True)
    pdf.cell(0, 8, f"Total Cleaned Messages: {total_cleaned:,}", ln=True)
    pdf.cell(0, 8, f"Removed Messages: {removed:,} ({removed_percent:.1f}%)", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Label Distribution:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Spam: {spam_count:,} ({spam_percent:.1f}%)", ln=True)
    pdf.cell(0, 8, f"  Ham: {ham_count:,} ({ham_percent:.1f}%)", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Message Statistics:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Average Word Count: {avg_words:.2f}", ln=True)
    pdf.cell(0, 8, f"  Average Character Count: {avg_chars:.2f}", ln=True)
    pdf.cell(0, 8, f"  Maximum Words: {max_words}", ln=True)
    pdf.cell(0, 8, f"  Minimum Words: {min_words}", ln=True)
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'Features Preserved:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, f"  Messages with URLs: {has_url:,}", ln=True)
    pdf.cell(0, 8, f"  Messages with Emails: {has_email:,}", ln=True)
    pdf.cell(0, 8, f"  Messages with Phone Numbers: {has_phone:,}", ln=True)
    pdf.ln(10)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '2. Data Cleaning Steps Performed', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    cleaning_steps = [
        {
            'step': 'Step 1: SMS Abbreviation Expansion',
            'desc': 'Expanded common SMS abbreviations (u->you, pls->please, etc.)'
        },
        {
            'step': 'Step 2: Punctuation Spacing Fix',
            'desc': 'Fixed spacing around punctuation marks for better readability'
        },
        {
            'step': 'Step 3: Extra Whitespace Removal',
            'desc': 'Removed multiple spaces and normalized whitespace'
        },
        {
            'step': 'Step 4: Repeated Punctuation Removal',
            'desc': 'Removed repeated punctuation marks (!!, ??, etc.)'
        },
        {
            'step': 'Step 5: Lowercase Conversion',
            'desc': 'Converted all text to lowercase for consistency'
        },
        {
            'step': 'Step 6: Empty Message Removal',
            'desc': f'Removed messages with no content or very short content ({removed:,} messages)'
        },
        {
            'step': 'Step 7: Duplicate Message Removal',
            'desc': 'Removed duplicate messages to avoid bias in training'
        },
        {
            'step': 'Step 8: URL Preservation',
            'desc': 'Kept URLs as is without modification (http, https, www)'
        },
        {
            'step': 'Step 9: Email Preservation',
            'desc': 'Kept email addresses as is without modification'
        },
        {
            'step': 'Step 10: Phone Number Preservation',
            'desc': 'Kept phone numbers as is without modification'
        }
    ]

    for item in cleaning_steps:
        pdf.set_font('Arial', 'B', 12)
        pdf.cell(0, 8, f"  {item['step']}", ln=True)
        pdf.set_font('Arial', '', 11)
        pdf.multi_cell(0, 6, f"      {item['desc']}")
        pdf.ln(2)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '3. Sample of Cleaned Data', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 10)
    sample = df_clean[['label', 'cleaned_message']].head(8)

    for idx, row in sample.iterrows():
        pdf.set_font('Arial', 'B', 11)
        pdf.cell(0, 7, f"Message {idx+1} - Label: {row['label'].upper()}", ln=True)
        pdf.set_font('Arial', '', 10)
        text = clean_text_for_pdf(row['cleaned_message'])
        if len(text) > 90:
            text = text[:90] + "..."
        pdf.multi_cell(0, 6, f"  {text}")
        pdf.cell(0, 4, "-" * 60, ln=True)
        pdf.ln(2)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '4. Output Files Generated', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'CSV File:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, "  File Name: cleaned_sms_dataset.csv", ln=True)
    pdf.cell(0, 8, "  Columns: 2", ln=True)
    pdf.cell(0, 8, "    - label: Class label (spam/ham)", ln=True)
    pdf.cell(0, 8, "    - cleaned_message: Preprocessed text message", ln=True)
    pdf.cell(0, 8, f"  Total Rows: {total_cleaned:,}", ln=True)
    pdf.cell(0, 8, "  File Location: /content/drive/MyDrive/8th sem major project/", ln=True)
    pdf.ln(10)

    pdf.set_font('Arial', 'B', 14)
    pdf.cell(0, 10, 'PDF Report:', ln=True)
    pdf.set_font('Arial', '', 12)
    pdf.cell(0, 8, "  File Name: cleaning_report.pdf", ln=True)
    pdf.cell(0, 8, "  Content: Complete cleaning report with statistics and steps", ln=True)
    pdf.ln(10)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '5. Ready for Machine Learning Training', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', '', 12)
    pdf.multi_cell(0, 8, "The cleaned dataset is now ready for machine learning training with 2 columns: label and cleaned_message.")
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 8, "Python Code Example for Model Training:", ln=True)
    pdf.set_font('Arial', '', 9)

    code_lines = [
        "# Load and prepare data",
        "import pandas as pd",
        "from sklearn.model_selection import train_test_split",
        "from sklearn.feature_extraction.text import TfidfVectorizer",
        "from sklearn.ensemble import RandomForestClassifier",
        "from sklearn.metrics import accuracy_score, classification_report",
        "",
        "# Load cleaned data",
        "df = pd.read_csv('cleaned_sms_dataset.csv')",
        "X = df['cleaned_message']",
        "y = df['label']",
        "",
        "# Vectorize text",
        "vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,3))",
        "X_vec = vectorizer.fit_transform(X)",
        "",
        "# Split data",
        "X_train, X_test, y_train, y_test = train_test_split(",
        "    X_vec, y, test_size=0.2, random_state=42, stratify=y",
        ")",
        "",
        "# Train model",
        "model = RandomForestClassifier(n_estimators=100, random_state=42)",
        "model.fit(X_train, y_train)",
        "",
        "# Evaluate",
        "y_pred = model.predict(X_test)",
        "print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')",
        "print(classification_report(y_test, y_pred))"
    ]

    for line in code_lines:
        pdf.cell(0, 5, line, ln=True)

    pdf.add_page()
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, '6. Summary of Cleaning Actions', ln=True)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(5)

    pdf.set_font('Arial', 'B', 11)
    pdf.cell(70, 8, "Action", border=1)
    pdf.cell(90, 8, "Details", border=1)
    pdf.cell(30, 8, "Count", border=1, ln=True)

    pdf.set_font('Arial', '', 10)
    summary_data = [
        ("Original Messages", f"{total_original:,}", ""),
        ("Cleaned Messages", f"{total_cleaned:,}", ""),
        ("Removed Messages", f"{removed:,}", f"{removed_percent:.1f}%"),
        ("Spam Messages", f"{spam_count:,}", f"{spam_percent:.1f}%"),
        ("Ham Messages", f"{ham_count:,}", f"{ham_percent:.1f}%"),
        ("URLs Preserved", f"{has_url:,}", ""),
        ("Emails Preserved", f"{has_email:,}", ""),
        ("Phones Preserved", f"{has_phone:,}", ""),
        ("Average Words", f"{avg_words:.2f}", ""),
        ("Average Characters", f"{avg_chars:.2f}", ""),
        ("Max Words", f"{max_words}", ""),
        ("Min Words", f"{min_words}", ""),
    ]

    for row in summary_data:
        pdf.cell(70, 7, str(row[0]), border=1)
        pdf.cell(90, 7, str(row[1]), border=1)
        pdf.cell(30, 7, str(row[2]), border=1, ln=True)

    pdf.ln(10)

    pdf.set_font('Arial', 'I', 10)
    pdf.cell(0, 10, 'Generated by SMS Detection System - Google Colab', ln=True, align='C')

    try:
        pdf.output('cleaning_report.pdf', 'F')
        print(f" PDF saved locally: cleaning_report.pdf")

        import shutil
        drive_path = '/content/drive/MyDrive/8th sem major project/cleaning_report.pdf'
        shutil.copy('cleaning_report.pdf', drive_path)
        print(f" PDF copied to Google Drive: {drive_path}")

        if os.path.exists('cleaning_report.pdf'):
            file_size = os.path.getsize('cleaning_report.pdf')
            print(f" File size: {file_size:,} bytes")

        return 'cleaning_report.pdf'

    except Exception as e:
        print(f" Error saving PDF: {e}")

        try:
            pdf.output('cleaning_report.pdf')
            print(f"✅ PDF saved locally (alternative method): cleaning_report.pdf")
            return 'cleaning_report.pdf'
        except:
            print(" Failed to save PDF")
            return None

pdf_path = create_pdf_report(final_df, df)

if pdf_path and os.path.exists(pdf_path):
    print(f"\n📄 PDF Report Details:")
    print(f"  Location: {pdf_path}")
    print(f"  Size: {os.path.getsize(pdf_path):,} bytes")
    print(f"\n PDF generated successfully!")
else:
    print("\n⚠️ PDF could not be generated. Please check the error messages above.")

    print("\n🔄 Trying to generate a simpler PDF...")
    try:
        from fpdf import FPDF
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font('Arial', 'B', 16)
        pdf.cell(0, 10, 'SMS Spam Detection - Cleaning Report', ln=True)
        pdf.set_font('Arial', '', 12)
        pdf.cell(0, 8, f'Total Messages: {len(final_df):,}', ln=True)
        pdf.cell(0, 8, f'Spam: {len(final_df[final_df["label"]=="spam"]):,}', ln=True)
        pdf.cell(0, 8, f'Ham: {len(final_df[final_df["label"]=="ham"]):,}', ln=True)
        pdf.cell(0, 8, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', ln=True)
        pdf.output('simple_report.pdf')
        print(" Simple PDF saved as: simple_report.pdf")

        import shutil
        drive_path = '/content/drive/MyDrive/8th sem major project/simple_report.pdf'
        shutil.copy('simple_report.pdf', drive_path)
        print(f" Simple PDF copied to Drive: {drive_path}")

        pdf_path = 'simple_report.pdf'

    except Exception as e2:
        print(f" Simple PDF also failed: {e2}")

📄 Generating PDF report...
✅ PDF saved locally: cleaning_report.pdf
✅ PDF copied to Google Drive: /content/drive/MyDrive/8th sem major project/cleaning_report.pdf
✅ File size: 6,020 bytes

📄 PDF Report Details:
  Location: cleaning_report.pdf
  Size: 6,020 bytes

✅ PDF generated successfully!


In [ ]:

from google.colab import files

print("📥 Download files:")

files.download('cleaned_sms_dataset.csv')

if pdf_path and os.path.exists(pdf_path):
    files.download(pdf_path)
elif os.path.exists('cleaning_report.pdf'):
    files.download('cleaning_report.pdf')

print(" Files downloaded!")


📥 Download files:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Files downloaded!


In [41]:

print("="*60)
print(" CLEANING COMPLETE")
print("="*60)
print(f" Dataset Statistics:")
print(f"  Total: {len(final_df):,} messages")
print(f"  Spam: {len(final_df[final_df['label']=='spam']):,}")
print(f"  Ham: {len(final_df[final_df['label']=='ham']):,}")
print(f"\n📁 Files Saved in Google Drive:")
print(f"  📄 /content/drive/MyDrive/8th sem major project/cleaned_sms_dataset.csv")
print(f"  📄 /content/drive/MyDrive/8th sem major project/cleaning_report.pdf")
print("="*60)

 CLEANING COMPLETE
 Dataset Statistics:


NameError: name 'final_df' is not defined

In [42]:
print(df.head())
print(df["label"].dtype)
print(df["label"].unique())

   label                                    cleaned_message
0      1  your opinion about me? 1. over to. jada 3. kus...
1      1  what's up? do you want me to come online? if y...
2      1                      so you are working overtime ?
3      1  also sir, i sent you an email about how to log...
4      0  please stay at home. to encourage the notion o...
int64
[1 0]


In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_sms_dataset.csv")
df["label"] = df["label"].map({
    "ham": 1,
    "spam": 0
})

print(df.head())
print(df.info())
print(df['label'].value_counts())

   label                                    cleaned_message
0      1  your opinion about me? 1. over to. jada 3. kus...
1      1  what's up? do you want me to come online? if y...
2      1                      so you are working overtime ?
3      1  also sir, i sent you an email about how to log...
4      0  please stay at home. to encourage the notion o...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841 entries, 0 to 9840
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   label            9841 non-null   int64 
 1   cleaned_message  9841 non-null   object
dtypes: int64(1), object(1)
memory usage: 153.9+ KB
None
label
1    8304
0    1537
Name: count, dtype: int64


In [2]:
print(df["label"].head())
print(df["label"].dtype)

X = df["cleaned_message"]
y = df["label"]

print(y.unique())
print(y.dtype)

0    1
1    1
2    1
3    1
4    0
Name: label, dtype: int64
int64
[1 0]
int64


In [3]:
from sklearn.model_selection import train_test_split

X = df["cleaned_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
print("After split:")
print(y_train.unique())
print(y_train.dtype)

After split:
[1 0]
int64


In [5]:
print(y_train.unique())
print(type(y_train))
print(y_train.dtype)

[1 0]
<class 'pandas.core.series.Series'>
int64


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [7]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (7872, 5000)
Testing TF-IDF shape: (1969, 5000)


In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

import numpy as np

In [11]:
models = {

    "Naive Bayes": MultinomialNB(),

    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000
    ),

    "Linear SVM": LinearSVC(
        random_state=42
    ),

    "Linear SVM (Balanced)": LinearSVC(
        random_state=42,
        class_weight="balanced"
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        eval_metric="logloss"
    )
}

In [28]:
print("=" * 60)
print("5-FOLD CROSS VALIDATION")
print("=" * 60)

cv_results = {}

for model_name, model in models.items():

    scores = cross_val_score(
        estimator=model,
        X=X_train_tfidf,
        y=y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    cv_results[model_name] = scores

    print(f"\n{model_name}")
    print("-" * 40)
    print("Fold Accuracies : ", np.round(scores, 4))
    print("Mean Accuracy   : ", round(scores.mean(), 4))
    print("Standard Dev.   : ", round(scores.std(), 4))

5-FOLD CROSS VALIDATION

Naive Bayes
----------------------------------------
Fold Accuracies :  [0.9683 0.9549 0.9619 0.9587 0.9657]
Mean Accuracy   :  0.9619
Standard Dev.   :  0.0048

Logistic Regression
----------------------------------------
Fold Accuracies :  [0.9486 0.9371 0.9473 0.9384 0.9517]
Mean Accuracy   :  0.9446
Standard Dev.   :  0.0058

Linear SVM
----------------------------------------
Fold Accuracies :  [0.9727 0.9625 0.9676 0.967  0.9714]
Mean Accuracy   :  0.9682
Standard Dev.   :  0.0036

Linear SVM (Balanced)
----------------------------------------
Fold Accuracies :  [0.9689 0.9606 0.9676 0.9657 0.9651]
Mean Accuracy   :  0.9656
Standard Dev.   :  0.0028

XGBoost
----------------------------------------
Fold Accuracies :  [0.946  0.9441 0.9473 0.9479 0.953 ]
Mean Accuracy   :  0.9477
Standard Dev.   :  0.003


In [29]:
cv_summary = []

for model_name, model in models.items():

    scores = cross_val_score(
        estimator=model,
        X=X_train_tfidf,
        y=y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    cv_results[model_name] = scores

    cv_summary.append({
        "Algorithm": model_name,
        "Mean CV Accuracy": round(scores.mean(), 4),
        "Std": round(scores.std(), 4)
    })

import pandas as pd

pd.DataFrame(cv_summary)

,Algorithm,Mean CV Accuracy,Std
0,Naive Bayes,0.9619,0.0048
1,Logistic Regression,0.9446,0.0058
2,Linear SVM,0.9682,0.0036
3,Linear SVM (Balanced),0.9656,0.0028
4,XGBoost,0.9477,0.0030


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

print("=" * 60)
print("HYPERPARAMETER TUNING - LINEAR SVM")
print("=" * 60)

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=LinearSVC(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_tfidf, y_train)

print("\nBest Parameters")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy")
print(round(grid_search.best_score_, 4))

HYPERPARAMETER TUNING - LINEAR SVM
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Best Parameters
{'C': 1}

Best Cross Validation Accuracy
0.9682


In [14]:
print("=" * 60)
print("ALL GRID SEARCH RESULTS")
print("=" * 60)

results = grid_search.cv_results_

for mean_score, std_score, params in zip(
    results["mean_test_score"],
    results["std_test_score"],
    results["params"]
):
    print(
        f"C={params['C']:<6} "
        f"Accuracy={mean_score:.4f} "
        f"Std={std_score:.4f}"
    )

ALL GRID SEARCH RESULTS
C=0.01   Accuracy=0.8445 Std=0.0007
C=0.1    Accuracy=0.9505 Std=0.0047
C=1      Accuracy=0.9682 Std=0.0036
C=10     Accuracy=0.9618 Std=0.0014
C=100    Accuracy=0.9538 Std=0.0016


In [16]:
final_svm_model = LinearSVC(
    C=1,
    random_state=42
)
final_svm_model = grid_search.best_estimator_

y_pred_final = final_svm_model.predict(X_test_tfidf)

In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print("Accuracy:", accuracy_score(y_test, y_pred_final))

print("\nClassification Report")
print(classification_report(y_test, y_pred_final))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_final))

FINAL MODEL EVALUATION
Accuracy: 0.9751142712036567

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.86      0.92       308
           1       0.98      1.00      0.99      1661

    accuracy                           0.98      1969
   macro avg       0.97      0.93      0.95      1969
weighted avg       0.98      0.98      0.97      1969


Confusion Matrix
[[ 266   42]
 [   7 1654]]


In [23]:
import joblib

final_svm_model = grid_search.best_estimator_

joblib.dump(final_svm_model, "../models/message/svm_message_model.pkl")
joblib.dump(vectorizer, "../models/message/tfidf_vectorizer.pkl")

print("Final model saved successfully.")

Final model saved successfully.


In [20]:
import os

print(os.getcwd())

/Users/binitapandey/phishing-detection-ml-system/notebooks


In [22]:
%who

GridSearchCV	 LinearSVC	 LogisticRegression	 MultinomialNB	 TfidfVectorizer	 X	 XGBClassifier	 X_test	 X_test_tfidf	 
X_train	 X_train_tfidf	 accuracy_score	 classification_report	 confusion_matrix	 cross_val_score	 cv_results	 df	 final_svm_model	 
grid_search	 joblib	 mean_score	 model	 model_name	 models	 np	 os	 param_grid	 
params	 pd	 results	 scores	 std_score	 train_test_split	 vectorizer	 y	 y_pred_final	 
y_test	 y_train	 


In [ ]:
##ALL the below code are implemented without crossfold

In [20]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_sms_dataset.csv")

In [21]:
print(df["label"].head())
print(df["label"].unique())

0     ham
1     ham
2     ham
3     ham
4    spam
Name: label, dtype: object
['ham' 'spam']


In [22]:
df["label"] = df["label"].map({
    "ham": 1,
    "spam": 0
})

print(df["label"].unique())

[1 0]


In [23]:
print(df["label"].unique())
print(y_train.unique())

[1 0]
[1 0]


In [24]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss"
)

xgb_model.fit(X_train_tfidf, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [25]:
print(df["label"].head(10))

0    1
1    1
2    1
3    1
4    0
5    0
6    1
7    1
8    1
9    1
Name: label, dtype: int64


In [26]:
print(df["label"].unique())
print(y_train.unique())
print(y_test.unique())

[1 0]
[1 0]
[1 0]


In [27]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric="logloss"
)

xgb_model.fit(X_train_tfidf, y_train)

y_pred_xgb = xgb_model.predict(X_test_tfidf)

print("===== XGBoost =====")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

===== XGBoost =====
Accuracy: 0.9517521584560691

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.71      0.82       308
           1       0.95      1.00      0.97      1661

    accuracy                           0.95      1969
   macro avg       0.96      0.85      0.90      1969
weighted avg       0.95      0.95      0.95      1969


Confusion Matrix:
[[ 218   90]
 [   5 1656]]


In [24]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_balanced = LinearSVC(
    random_state=42,
    class_weight="balanced"
)

svm_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = svm_balanced.predict(X_test_tfidf)

print("===== Linear SVM (Balanced) =====")
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

===== Linear SVM (Balanced) =====
Accuracy: 0.9680040629761301

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       308
           1       0.98      0.98      0.98      1661

    accuracy                           0.97      1969
   macro avg       0.94      0.94      0.94      1969
weighted avg       0.97      0.97      0.97      1969


Confusion Matrix:
[[ 275   33]
 [  30 1631]]


In [29]:
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

cnb = ComplementNB()

cnb.fit(X_train_tfidf, y_train)

y_pred = cnb.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9319451498222447
              precision    recall  f1-score   support

           0       0.72      0.91      0.81       308
           1       0.98      0.94      0.96      1661

    accuracy                           0.93      1969
   macro avg       0.85      0.92      0.88      1969
weighted avg       0.94      0.93      0.94      1969

[[ 281   27]
 [ 107 1554]]


In [30]:
import joblib

In [31]:
joblib.dump(
    svm_model,
    "../models/message/svm_message_model.pkl"
)

print("Linear SVM model saved successfully!")

Linear SVM model saved successfully!


In [32]:
joblib.dump(
    vectorizer,
    "../models/message/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer saved successfully!")

TF-IDF Vectorizer saved successfully!


In [33]:
loaded_model = joblib.load("../models/message/svm_message_model.pkl")

loaded_vectorizer = joblib.load("../models/message/tfidf_vectorizer.pkl")

print("Model loaded successfully!")
print("Vectorizer loaded successfully!")

Model loaded successfully!
Vectorizer loaded successfully!


In [34]:
sample_message = [
    "Congratulations! You have won a $1000 prize. Click here to claim now."
]

sample_vector = loaded_vectorizer.transform(sample_message)

prediction = loaded_model.predict(sample_vector)

print(prediction)

[0]


In [35]:
label = "Legitimate" if prediction[0] == 1 else "Phishing"

print(label)

Phishing


In [36]:
df[df["cleaned_message"].str.contains("congrat", case=False, na=False)][["label", "cleaned_message"]].head(20)

,label,cleaned_message
193,0,congratulations your awarded either £500 of cd...
605,0,congrats! 1 year special cinema pass for to is...
706,1,ok�congrats�
1009,0,congratulations your awarded either £500 of cd...
1363,1,prof: you have passed in all the papers in thi...
1429,0,"congratulations, your entry into our contest l..."
1520,0,congratulation! your mobile no was selected as...
1594,1,congratulations ore mo owo re wa. enjoy it and...
1764,0,congrats! to mobile 3g videophones are yours. ...
1792,1,okay:)all the best:)congrats.


In [37]:
df[df["cleaned_message"].str.contains("won", case=False, na=False)][["label", "cleaned_message"]].head(20)

,label,cleaned_message
38,0,you have won! as a valued vodafone customer ou...
56,1,kallis wont bat in 2nd innings.
72,1,love it! the girls at the office may wonder wh...
86,0,hmv bonus special 500 pounds of genuine hmv vo...
92,0,you have won a guaranteed £200 award or even £...
116,0,this is the 2nd time we have tried to contact ...
126,1,i'm really sorry i won't be able to do this fr...
239,1,laugh out loud yes. our friendship is hanging ...
366,1,no. thank you. you've been wonderful
378,0,you have won a guaranteed £1000 cash or a £200...


In [38]:
df[df["cleaned_message"].str.contains("click", case=False, na=False)][["label", "cleaned_message"]].head(20)

,label,cleaned_message
131,0,pl: battle royale! gilchrist vs warne today at...
467,0,"xxxmobilemovieclub: to use your credit, click ..."
1159,0,we have recalculated your vehicle tax. you are...
1314,0,(help-center canada) you received a transfer f...
1687,0,welcome to paytm payment bank your re verifica...
2050,0,"update message to user,log on to verity your l..."
2151,0,mobile internet ka full fayda. ab karo downloa...
2153,1,yeah just open chat and click friend lists. th...
2570,0,need to clear doubts before exams? experienced...
2896,0,"collect 100 points& get 14,500 pokecoins for p..."


In [39]:
df[df["cleaned_message"].str.contains("claim", case=False, na=False)][["label", "cleaned_message"]].head(20)

,label,cleaned_message
42,0,phony £350 award - todays voda numbers ending ...
54,0,todays voda numbers ending 7548 are selected t...
92,0,you have won a guaranteed £200 award or even £...
116,0,this is the 2nd time we have tried to contact ...
246,0,urgent! your mobile number has been awarded wi...
255,0,todays voda numbers ending 5226 are selected t...
258,0,winner! as a valued network customer you have ...
270,0,"urgent! dear fl1pkart customer, we are trying ..."
284,0,"dear voucher holder, to claim this weeks offer..."
307,0,you are a winner you have been specially selec...


In [40]:
type(svm_model)

sklearn.svm._classes.LinearSVC

In [41]:
print(type(svm_model))

<class 'sklearn.svm._classes.LinearSVC'>


In [42]:
print(vectorizer.vocabulary_.get("congratulations"))

899


In [43]:
feature_names = vectorizer.get_feature_names_out()

coef = svm_model.coef_[0]

top_spam = sorted(zip(coef, feature_names), reverse=True)[:30]
top_ham = sorted(zip(coef, feature_names))[:30]

print("Top Spam Words")
for w, f in top_spam:
    print(f"{f:20} {w:.3f}")

print("\nTop Ham Words")
for w, f in top_ham:
    print(f"{f:20} {w:.3f}")

Top Spam Words
नह                   1.184
liked                1.072
later                1.020
घर                   1.002
अच                   0.971
बत                   0.933
gt                   0.931
fullonsms com        0.885
fullonsms            0.885
कह                   0.881
hey                  0.858
गलत                  0.846
mail id              0.801
आम                   0.800
वह                   0.791
je                   0.786
ich                  0.779
यह एक                0.762
तरह                  0.735
said                 0.723
sure                 0.721
ok                   0.718
lt                   0.716
wgt                  0.713
हम इस                0.709
mein                 0.709
cool                 0.685
अपन आप               0.682
cheers               0.675
battery              0.668

Top Ham Words
www                  -2.657
http                 -2.589
uk                   -2.407
150p                 -2.164
50                   -2.013
mobile              

In [44]:
print(df.head())
print(df.tail())
print(df.sample(10))

   label                                    cleaned_message
0      1  your opinion about me? 1. over to. jada 3. kus...
1      1  what's up? do you want me to come online? if y...
2      1                      so you are working overtime ?
3      1  also sir, i sent you an email about how to log...
4      0  please stay at home. to encourage the notion o...
      label                                    cleaned_message
9836      0  diese nachricht wird ihnen von gmw ltd. gebrac...
9837      0  willst du to heute nacht flachgelegt? willst d...
9838      1                  moji je t'aime plus que les mots.
9839      1    तो मैं तुम्हें अपना हिसाब संख्या भेजने वाला हूँ
9840      1           unni merci cher pour la recharge.rakhesh
      label                                    cleaned_message
3292      1  ard 530 like dat lor. we juz meet in mrt stati...
3757      1                  almost there, see you in a second
1307      1  you lifted my hopes with the offer of money. i...
5466      

In [45]:
feature_names = vectorizer.get_feature_names_out()
coef = svm_model.coef_[0]

weights = pd.DataFrame({
    "word": feature_names,
    "weight": coef
})

print(weights.sort_values("weight", ascending=False).head(30))
print(weights.sort_values("weight").head(30))

               word    weight
4830             नह  1.183501
2327          liked  1.072474
2244          later  1.020296
4768             घर  1.001981
4595             अच  0.971070
4869             बत  0.933371
1706             gt  0.930737
1555  fullonsms com  0.885005
1554      fullonsms  0.885005
4747             कह  0.880741
1834            hey  0.858477
4766            गलत  0.845587
2444        mail id  0.800886
4649             आम  0.799820
4947             वह  0.790964
2079             je  0.786303
1931            ich  0.778586
4902          यह एक  0.762198
4809            तरह  0.735321
3367           said  0.723070
3774           sure  0.720963
2840             ok  0.717611
2405             lt  0.716092
4369            wgt  0.712540
4995          हम इस  0.709065
2510           mein  0.708886
925            cool  0.684722
4605         अपन आप  0.681912
794          cheers  0.674953
552         battery  0.668124
          word    weight
4486       www -2.656621
1904      http -2.58

In [46]:
print(svm_model.classes_)

[0 1]


In [47]:
words = [
    "congratulations",
    "won",
    "1000",
    "click",
    "claim",
    "free",
    "urgent",
    "winner"
]

for word in words:
    if word in vectorizer.vocabulary_:
        idx = vectorizer.vocabulary_[word]
        print(word, svm_model.coef_[0][idx])
    else:
        print(word, "NOT FOUND")

congratulations -0.8907779695188827
won -0.5322189524528356
1000 -1.4119370885825828
click -0.8597102820837945
claim -1.7311332256638607
free -0.8119913898393114
urgent -0.8914782721003259
winner -0.8299040449499262


In [48]:
msg = "Congratulations! You won $1000."

X = vectorizer.transform([msg])

print("Prediction:", svm_model.predict(X))
print("Decision score:", svm_model.decision_function(X))

Prediction: [0]
Decision score: [-0.37343106]


In [49]:
msg = "The official website is"
X = vectorizer.transform([msg])

print(svm_model.predict(X))
print(svm_model.decision_function(X))

[1]
[1.06902599]


In [50]:
msg = "The official website is https://www.apple.com"

X = vectorizer.transform([msg])

print(msg)
print(svm_model.predict(X))
print(svm_model.decision_function(X))

The official website is https://www.apple.com
[0]
[-0.94912787]


In [53]:
msg = "to learn more about Google's services."
X = vectorizer.transform([msg])

print(svm_model.predict(X))
print(svm_model.decision_function(X))

[0]
[-0.11129234]


In [54]:
feature_names = vectorizer.get_feature_names_out()
coef = svm_model.coef_[0]

for idx in X.nonzero()[1]:
    print(feature_names[idx], coef[idx])

google 0.35214853587562867
learn -0.7432465957284845
services -1.4860303811473168


In [55]:
df[df["cleaned_message"].str.contains("services", case=False, na=False)][["label","cleaned_message"]]

,label,cleaned_message
187,0,here is your discount code rp176781. to stop f...
231,0,"thanks for your ringtone order, reference numb..."
331,0,you are now unsubscribed all services. get ton...
785,0,important message from westpac for the safety ...
803,0,"thanks for your ringtone order, reference numb..."
1281,0,"thanks for your ringtone order, reference numb..."
1524,0,88800 and 89034 are premium phone services cal...
1667,0,sms services for your inclusive text credits p...
2130,0,your unique user id is 1172. for removal send ...
2347,0,"your iphone id is due to expire today, please ..."


In [56]:
spam_services = df[
    (df["label"] == "spam") &
    (df["cleaned_message"].str.contains("services", case=False, na=False))
].shape[0]

ham_services = df[
    (df["label"] == "ham") &
    (df["cleaned_message"].str.contains("services", case=False, na=False))
].shape[0]

print("Spam:", spam_services)
print("Ham :", ham_services)

Spam: 0
Ham : 0


In [57]:
print("services" in vectorizer.vocabulary_)
print(vectorizer.vocabulary_.get("services"))

True
3472
